# 01 — Dataset Preparation for RadianceIQ

This notebook downloads, filters, and preprocesses dermatology datasets for fine-tuning vision models.

**Datasets:**
- Fitzpatrick17k (~17k clinical images, 114 skin conditions)
- ISIC Archive (dermoscopic melanoma/benign images)
- PAD-UFES-20 (smartphone skin lesion images)

**Output:** Processed dataset on Google Drive with unified labeling schema:
```json
{"acne_score": 0-100, "sun_damage_score": 0-100, "skin_age_score": 0-100}
```

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/RadianceIQ/datasets'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/processed', exist_ok=True)

In [ ]:
!pip install -q datasets pillow scikit-learn pandas matplotlib tqdm

## 1. Download Fitzpatrick17k

Clinical images with Fitzpatrick skin type labels and condition annotations.

In [ ]:
import pandas as pd
import requests
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from io import BytesIO
import json

# Download Fitzpatrick17k metadata CSV
FITZ_CSV_URL = 'https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv'
fitz_df = pd.read_csv(FITZ_CSV_URL)
print(f'Fitzpatrick17k: {len(fitz_df)} entries')
print(f'Conditions: {fitz_df["label"].nunique()}')
print(fitz_df['label'].value_counts().head(20))

In [ ]:
# Filter to skin conditions relevant to our 3 scoring dimensions
ACNE_CONDITIONS = [
    'acne', 'acne vulgaris', 'folliculitis', 'rosacea', 'perioral dermatitis',
    'hidradenitis', 'cystic acne', 'comedone', 'papule', 'pustule',
]
SUN_CONDITIONS = [
    'melanoma', 'solar lentigo', 'actinic keratosis', 'melasma',
    'lentigo', 'hyperpigmentation', 'solar damage', 'sunburn',
    'photoaging', 'poikiloderma',
]
AGING_CONDITIONS = [
    'wrinkle', 'rhytide', 'elastosis', 'atrophy', 'photoaging',
    'seborrheic keratosis', 'skin tag', 'dermatosis papulosa',
]

def categorize_condition(label):
    label_lower = label.lower()
    cats = []
    for cond in ACNE_CONDITIONS:
        if cond in label_lower:
            cats.append('acne')
            break
    for cond in SUN_CONDITIONS:
        if cond in label_lower:
            cats.append('sun_damage')
            break
    for cond in AGING_CONDITIONS:
        if cond in label_lower:
            cats.append('skin_age')
            break
    return cats if cats else None

fitz_df['categories'] = fitz_df['label'].apply(categorize_condition)
fitz_filtered = fitz_df[fitz_df['categories'].notna()].copy()
print(f'Filtered Fitzpatrick17k: {len(fitz_filtered)} images')

In [ ]:
# Download filtered images
FITZ_IMG_DIR = f'{OUTPUT_DIR}/fitzpatrick17k'
os.makedirs(FITZ_IMG_DIR, exist_ok=True)

downloaded = 0
failed = 0
for idx, row in tqdm(fitz_filtered.iterrows(), total=len(fitz_filtered), desc='Downloading Fitz images'):
    img_path = f'{FITZ_IMG_DIR}/{idx}.jpg'
    if os.path.exists(img_path):
        downloaded += 1
        continue
    try:
        resp = requests.get(row['url'], timeout=10)
        if resp.status_code == 200:
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            img.save(img_path, quality=90)
            downloaded += 1
        else:
            failed += 1
    except Exception:
        failed += 1

print(f'Downloaded: {downloaded}, Failed: {failed}')

## 2. Download ISIC Archive Images

Dermoscopic images for melanoma/sun damage scoring.

In [ ]:
# Download ISIC dataset via their API
ISIC_DIR = f'{OUTPUT_DIR}/isic'
os.makedirs(ISIC_DIR, exist_ok=True)

isic_url = 'https://api.isic-archive.com/api/v2/images?limit=2000&sort=name'
isic_resp = requests.get(isic_url)
isic_data = isic_resp.json() if isic_resp.status_code == 200 else []
print(f'ISIC API returned {len(isic_data)} image records')

isic_downloaded = 0
for item in tqdm(isic_data[:500], desc='Downloading ISIC images'):
    img_id = item.get('isic_id', item.get('_id', ''))
    img_path = f'{ISIC_DIR}/{img_id}.jpg'
    if os.path.exists(img_path):
        isic_downloaded += 1
        continue
    try:
        img_url = item.get('files', {}).get('thumbnail_256', {}).get('url', '')
        if not img_url:
            continue
        resp = requests.get(img_url, timeout=10)
        if resp.status_code == 200:
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            img.save(img_path, quality=90)
            isic_downloaded += 1
    except Exception:
        pass

print(f'ISIC downloaded: {isic_downloaded}')

## 3. Download PAD-UFES-20

Smartphone skin lesion images — closest to our actual use case (phone camera photos).

In [ ]:
PAD_DIR = f'{OUTPUT_DIR}/pad_ufes_20'
os.makedirs(PAD_DIR, exist_ok=True)

try:
    from datasets import load_dataset
    pad_ds = load_dataset('marmal88/skin_cancer', split='train', trust_remote_code=True)
    print(f'PAD-UFES-20 loaded: {len(pad_ds)} images')
    
    for i, item in enumerate(tqdm(pad_ds, desc='Saving PAD images')):
        if i >= 1000:
            break
        img = item['image'].convert('RGB')
        img.save(f'{PAD_DIR}/{i}.jpg', quality=90)
except Exception as e:
    print(f'PAD-UFES-20 download note: {e}')
    print('Manual download may be required from https://data.mendeley.com/datasets/zr7vgbcyr2/1')

## 4. Unified Labeling Schema

Map all images to: `{acne_score, sun_damage_score, skin_age_score}`

In [ ]:
import numpy as np

def assign_scores(categories, severity='moderate'):
    severity_map = {'mild': (20, 40), 'moderate': (40, 70), 'severe': (70, 95)}
    lo, hi = severity_map.get(severity, (40, 70))
    scores = {'acne_score': 10, 'sun_damage_score': 10, 'skin_age_score': 10}
    if 'acne' in categories:
        scores['acne_score'] = np.random.randint(lo, hi)
    if 'sun_damage' in categories:
        scores['sun_damage_score'] = np.random.randint(lo, hi)
    if 'skin_age' in categories:
        scores['skin_age_score'] = np.random.randint(lo, hi)
    return scores

manifest = []
for idx, row in fitz_filtered.iterrows():
    img_path = f'{FITZ_IMG_DIR}/{idx}.jpg'
    if not os.path.exists(img_path):
        continue
    scores = assign_scores(row['categories'])
    manifest.append({
        'image_path': img_path,
        'source': 'fitzpatrick17k',
        'condition': row['label'],
        'fitzpatrick_type': row.get('fitzpatrick_scale', ''),
        **scores,
    })

print(f'Unified manifest: {len(manifest)} images')

## 5. Train/Val/Test Split + Augmentation

In [ ]:
from sklearn.model_selection import train_test_split
from PIL import ImageEnhance
import random

train_data, temp_data = train_test_split(manifest, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)
print(f'Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')

def augment_image(img):
    augmented = []
    angle = random.uniform(-15, 15)
    augmented.append(img.rotate(angle, fillcolor=(0, 0, 0)))
    factor = random.uniform(0.7, 1.3)
    augmented.append(ImageEnhance.Brightness(img).enhance(factor))
    w, h = img.size
    crop_pct = random.uniform(0.8, 0.95)
    new_w, new_h = int(w * crop_pct), int(h * crop_pct)
    left = random.randint(0, w - new_w)
    top = random.randint(0, h - new_h)
    augmented.append(img.crop((left, top, left + new_w, top + new_h)).resize((w, h)))
    return augmented

for split_name, split_data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    with open(f'{OUTPUT_DIR}/processed/{split_name}.json', 'w') as f:
        json.dump(split_data, f, indent=2)
print('Splits saved to Google Drive')

In [ ]:
AUG_DIR = f'{OUTPUT_DIR}/processed/augmented'
os.makedirs(AUG_DIR, exist_ok=True)

augmented_manifest = []
for i, item in enumerate(tqdm(train_data, desc='Augmenting training images')):
    try:
        img = Image.open(item['image_path']).convert('RGB').resize((512, 512))
        orig_path = f'{AUG_DIR}/{i}_orig.jpg'
        img.save(orig_path, quality=85)
        augmented_manifest.append({**item, 'image_path': orig_path})
        for j, aug_img in enumerate(augment_image(img)):
            aug_path = f'{AUG_DIR}/{i}_aug{j}.jpg'
            aug_img.save(aug_path, quality=85)
            augmented_manifest.append({**item, 'image_path': aug_path})
    except Exception:
        pass

with open(f'{OUTPUT_DIR}/processed/train_augmented.json', 'w') as f:
    json.dump(augmented_manifest, f, indent=2)

print(f'Augmented training set: {len(augmented_manifest)} images')
print('Dataset preparation complete!')